# Arsitrad Full - Kaggle Technical Notebook

This is the end-to-end Kaggle notebook for Arsitrad v2: **AI Advisor / Building Doctor** for Indonesian architecture and construction regulation.

Use Kaggle with Internet ON and a GPU runtime enabled. T4 is the target.

Run every cell in order. This notebook is opinionated on purpose: local GGUF install, Gemma vision projector download, answer generation, evaluation dry-run, and the Next.js + FastAPI demo launch are part of the flow, not side quests.

Important: unlike the lightweight demo notebook, this full notebook tracks the latest `main` branch because the Kaggle GGUF runtime, Building Doctor vision bridge, and web demo hardening live there, not in the frozen `v2.0.0` release tag.

What it covers:
- clone the latest `main` repo state
- install notebook dependencies plus a prebuilt CUDA `llama-cpp-python` wheel
- install the Next.js frontend dependencies
- inspect corpus and config
- bootstrap sparse snapshot automatically if the JSONL artifact is missing
- run dry-run ingest on a subset
- inspect sparse retrieval directly
- run GGUF answer generation through the real Arsitrad engine
- download the Gemma `mmproj` vision projector for visual triage
- run the starter evaluation flow
- boot Gemma vision bridge + FastAPI + Next.js and expose the AI Advisor workbench through Cloudflare from inside Kaggle

For a faster, lower-drama demo, use `Arsitrad_v2_Kaggle_Demo.ipynb` instead.

## Notebook posture

This notebook now runs the portable Arsitrad path end to end:
- sparse snapshot bootstrap
- retrieval inspection
- GGUF answer generation
- evaluation dry-run
- Gemma vision bridge for Building Doctor visual triage
- FastAPI backend launch with `ARSITRAD_VISION_BASE_URL` already set
- Next.js AI Advisor frontend launch
- Cloudflare public URL exposure for the web workbench

If you also want full hybrid retrieval, set `ARSITRAD_DATABASE_URL` before the retrieval or answer cells. The notebook will pick it up automatically, but the base sequential run no longer asks you to choose between manual branches.

In [ ]:
%cd /kaggle/working
!rm -rf arsitrad
!git clone --depth 1 --branch main https://github.com/arsitekberotot/arsitrad.git
%cd /kaggle/working/arsitrad
!git rev-parse --short HEAD
from pathlib import Path
config_text = Path('config.yaml').read_text(encoding='utf-8')
inference_text = Path('pipeline/inference.py').read_text(encoding='utf-8')
assert 'n_gpu_layers' in config_text, 'Cloned repo is missing Kaggle GGUF runtime config fields.'
assert 'ARSITRAD_GGUF_N_GPU_LAYERS' in inference_text, 'Cloned repo is missing GGUF env override support.'
print('Repo sanity check passed for Kaggle GGUF runtime.')


In [ ]:
import importlib.util
import subprocess
import sys

base_packages = [
    'pyyaml',
    'pdfplumber',
    'rank-bm25',
    'requests',
    'pydantic',
    'huggingface_hub',
    'fastapi',
    'uvicorn',
    'pytest',
]

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--upgrade-strategy',
    'only-if-needed',
    *base_packages,
])

gpu_probe = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_probe.returncode != 0:
    raise RuntimeError('This notebook expects a Kaggle GPU runtime. Enable T4 before continuing.')
print(gpu_probe.stdout.splitlines()[0])

llama_wheel_url = 'https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.20-cu124/llama_cpp_python-0.3.20-py3-none-linux_x86_64.whl'
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--no-cache-dir',
    '--force-reinstall',
    llama_wheel_url,
])

import numpy as np
import pandas as pd
from llama_cpp import Llama

print('Using platform NumPy version:', np.__version__)
print('Using platform pandas version:', pd.__version__)
print('llama_cpp installed:', importlib.util.find_spec('llama_cpp') is not None)
print('Llama import OK:', Llama is not None)
print('Installed prebuilt CUDA wheel:', llama_wheel_url)


## Runtime assumptions

- This notebook installs `llama-cpp-python` and expects the GGUF download step to succeed later.
- Sparse retrieval works from repo assets alone.
- Full hybrid retrieval still needs your own `ARSITRAD_DATABASE_URL`, but that is an external database prerequisite, not a notebook branch you have to choose manually.


In [ ]:
import json
import os
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import yaml

config = yaml.safe_load((repo_root / 'config.yaml').read_text(encoding='utf-8'))
print(json.dumps(config['v2'], ensure_ascii=False, indent=2)[:4000])


In [ ]:
import importlib
import json
import os
import sys
from collections import Counter
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
importlib.invalidate_caches()

from pipeline.chunker import infer_number, infer_region, infer_year
from pipeline.taxonomy import enrich_metadata, infer_building_use, infer_topic

text_paths = sorted((repo_root / 'data/corpus').rglob('*.txt'))
corpus_path = repo_root / 'data/processed/v2/bm25_corpus.jsonl'

def build_sparse_snapshot_from_tracked_chunks(output_path: Path) -> int:
    chunk_sources = [
        (repo_root / 'data/processed/national_chunks.json', 'national'),
        (repo_root / 'data/processed/local_chunks.json', 'local'),
    ]
    records = []
    for chunk_path, scope in chunk_sources:
        if not chunk_path.exists():
            continue
        payload = json.loads(chunk_path.read_text(encoding='utf-8'))
        chunks = payload.get('chunks', [])
        metadatas = payload.get('metadatas', [])
        ids = payload.get('ids', [])
        for chunk_text, meta, chunk_id in zip(chunks, metadatas, ids):
            source_name = str(meta.get('source') or meta.get('title') or chunk_id)
            is_local = scope == 'local'
            reg_type = str(meta.get('reg_type') or ('Perda' if is_local else 'Unknown')).title()
            year = meta.get('year') or infer_year(source_name)
            number = meta.get('number') or infer_number(source_name)
            region = 'nasional' if not is_local else infer_region(Path(source_name + '.txt'), source_name, is_local=True)
            metadata = {
                'source_name': source_name,
                'source_path': str((Path('/data/corpus/local_regulations') if is_local else Path('/data/corpus/indonesian-construction-law')) / f'{source_name}.txt'),
                'source_file': f'{source_name}.txt',
                'reg_type': reg_type,
                'year': year,
                'number': number,
                'region': region,
                'scope': 'local' if is_local else 'national',
                'island': meta.get('island'),
                'chunk_index': int(meta.get('chunk_idx', 0) or 0),
                'start_page': int(meta.get('page', 1) or 1),
                'end_page': int(meta.get('page', 1) or 1),
                'title': source_name,
            }
            topic = infer_topic(source_name, chunk_text)
            building_use = infer_building_use(source_name, chunk_text)
            if topic:
                metadata['topic'] = topic
                metadata['typology'] = topic
            if building_use:
                metadata['building_use'] = building_use
            metadata = enrich_metadata(metadata, str(chunk_text))
            records.append({
                'chunk_key': str(chunk_id),
                'content': str(chunk_text),
                'metadata': metadata,
                'start_page': metadata['start_page'],
                'end_page': metadata['end_page'],
            })
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', encoding='utf-8') as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    return len(records)

needs_bootstrap = True
if corpus_path.exists():
    row_count = sum(1 for line in corpus_path.open('r', encoding='utf-8') if line.strip())
    if row_count > 0:
        needs_bootstrap = False
        print('Using existing sparse corpus snapshot:', corpus_path)
    else:
        print('Sparse snapshot exists but is empty. Rebuilding it from tracked chunk stores...')
if needs_bootstrap:
    print('Sparse snapshot missing. Bootstrapping from tracked chunk stores...')
    built = build_sparse_snapshot_from_tracked_chunks(corpus_path)
    print('Bootstrapped sparse records:', built)

source_counts = Counter()
row_count = 0
with corpus_path.open('r', encoding='utf-8') as handle:
    for line in handle:
        record = json.loads(line)
        row_count += 1
        metadata = record.get('metadata') or {}
        source_counts[metadata.get('source_name', 'UNKNOWN')] += 1

print('Text documents  =', len(text_paths))
print('Sparse rows     =', row_count)
print('Top sources:')
for name, count in source_counts.most_common(10):
    print(f'  - {name}: {count}')


In [ ]:
%cd /kaggle/working/arsitrad
!python -m pipeline.ingest --config config.yaml --dry-run --from-sparse --limit-docs 5


In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
importlib.invalidate_caches()

from pipeline.retriever import HybridRetriever

retriever = HybridRetriever(config_path=str(repo_root / 'config.yaml'))

queries = [
    'Apa bedanya IMB dan PBG?',
    'Apakah RDTR wajib dicek sebelum mengurus PBG?',
    'Apa yang harus dicek untuk bangunan di dekat sungai menurut tata ruang?',
]

for query in queries:
    result = retriever.retrieve(query)
    print('=' * 100)
    print('QUESTION:', query)
    print('STANDALONE:', result.standalone_query)
    print('CONFIDENCE:', f'{result.confidence:.2f}')
    print('SHOULD_ANSWER:', result.should_answer)
    for idx, candidate in enumerate(result.candidates[:3], start=1):
        print(f'  [{idx}] {candidate.metadata.get("source_name", "UNKNOWN")} | region={candidate.metadata.get("region", "-")} | score={candidate.score:.4f}')


## GGUF answer engine

This notebook now treats GGUF generation as a required step.
The next cell downloads the model, and the following cell must answer with `used_model = True`.


In [ ]:
import importlib.util
import os
import sys
from pathlib import Path
from huggingface_hub import hf_hub_download

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

model_dir = repo_root / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / 'gemma-4-E4B-it-Q4_K_M.gguf'
mmproj_path = model_dir / 'mmproj-gemma-4-E4B-it-Q8_0.gguf'

if importlib.util.find_spec('llama_cpp') is None:
    raise RuntimeError('llama_cpp is missing. Step 2 was supposed to install the CUDA wheel for this notebook.')

os.environ.update({
    'ARSITRAD_GGUF_MODEL_PATH': str(model_path),
    'ARSITRAD_GGUF_CONTEXT_WINDOW': '4096',
    'ARSITRAD_GGUF_N_GPU_LAYERS': '-1',
    'ARSITRAD_GGUF_N_THREADS': '2',
    'ARSITRAD_GGUF_N_BATCH': '256',
    'ARSITRAD_VISION_MODEL_PATH': str(model_path),
    'ARSITRAD_VISION_MMPROJ_PATH': str(mmproj_path),
    'ARSITRAD_VISION_BASE_URL': 'http://127.0.0.1:8080',
    'ARSITRAD_VISION_MODEL': 'gemma-4-E4B-it',
    'ARSITRAD_VISION_MAX_TOKENS': '220',
    'ARSITRAD_VISION_TIMEOUT_SECONDS': '60',
    'ARSITRAD_REQUIRE_VISION': '1',
})

gguf_path = hf_hub_download(
    repo_id='ggml-org/gemma-4-E4B-it-GGUF',
    filename='gemma-4-E4B-it-Q4_K_M.gguf',
    local_dir=str(model_dir),
)

mmproj_download_path = hf_hub_download(
    repo_id='ggml-org/gemma-4-E4B-it-GGUF',
    filename='mmproj-gemma-4-E4B-it-Q8_0.gguf',
    local_dir=str(model_dir),
)

if not model_path.exists():
    raise FileNotFoundError(model_path)
if not mmproj_path.exists():
    raise FileNotFoundError(mmproj_path)

size_gb = model_path.stat().st_size / (1024 ** 3)
mmproj_size_gb = mmproj_path.stat().st_size / (1024 ** 3)
print('GGUF ready at:', gguf_path)
print(f'GGUF size: {size_gb:.2f} GB')
print('Gemma vision projector ready at:', mmproj_download_path)
print(f'Vision projector size: {mmproj_size_gb:.2f} GB')
print('Runtime overrides:')
for name in [
    'ARSITRAD_GGUF_MODEL_PATH',
    'ARSITRAD_GGUF_CONTEXT_WINDOW',
    'ARSITRAD_GGUF_N_GPU_LAYERS',
    'ARSITRAD_GGUF_N_THREADS',
    'ARSITRAD_GGUF_N_BATCH',
    'ARSITRAD_VISION_MODEL_PATH',
    'ARSITRAD_VISION_MMPROJ_PATH',
    'ARSITRAD_VISION_BASE_URL',
    'ARSITRAD_REQUIRE_VISION',
]:
    print(f'  {name}={os.environ[name]}')

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

model_path = repo_root / 'models' / 'gemma-4-E4B-it-Q4_K_M.gguf'
os.environ.update({
    'ARSITRAD_GGUF_MODEL_PATH': str(model_path),
    'ARSITRAD_GGUF_CONTEXT_WINDOW': '4096',
    'ARSITRAD_GGUF_N_GPU_LAYERS': '-1',
    'ARSITRAD_GGUF_N_THREADS': '2',
    'ARSITRAD_GGUF_N_BATCH': '256',
})

from pipeline.inference import ArsitradAnswerEngine, load_inference_config

config_path = repo_root / 'config.yaml'
cfg = load_inference_config(config_path)
print('Loaded GGUF config:', cfg)
if not Path(cfg.model_path).exists():
    raise FileNotFoundError(cfg.model_path)
if cfg.n_gpu_layers != -1:
    raise RuntimeError(f'Expected full GPU offload on Kaggle, got n_gpu_layers={cfg.n_gpu_layers}')

engine = ArsitradAnswerEngine(config_path=config_path)
result = engine.answer('Apa syarat PBG untuk rumah tinggal 2 lantai di Semarang?')
print('used_model =', result.used_model)
if not result.used_model:
    raise RuntimeError('Expected a GGUF-backed answer, but Arsitrad fell back instead.')
print('confidence =', result.retrieval.confidence)
print(result.answer[:2500])


In [ ]:
%cd /kaggle/working/arsitrad
!python -m pipeline.eval.ragas_eval --questions data/eval/golden_queries.json --output /kaggle/working/arsitrad_eval.json --dry-run

In [ ]:
import json
import os
import re
import shutil
import signal
import stat
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
os.chdir(repo_root)
model_path = Path(os.environ.get('ARSITRAD_VISION_MODEL_PATH', repo_root / 'models' / 'gemma-4-E4B-it-Q4_K_M.gguf'))
mmproj_path = Path(os.environ.get('ARSITRAD_VISION_MMPROJ_PATH', repo_root / 'models' / 'mmproj-gemma-4-E4B-it-Q8_0.gguf'))
vision_url = os.environ.get('ARSITRAD_VISION_BASE_URL', 'http://127.0.0.1:8080').rstrip('/')
require_vision = os.environ.get('ARSITRAD_REQUIRE_VISION', '1') == '1'
os.environ.update({
    'ARSITRAD_GGUF_MODEL_PATH': str(model_path),
    'ARSITRAD_GGUF_CONTEXT_WINDOW': '4096',
    'ARSITRAD_GGUF_N_GPU_LAYERS': '-1',
    'ARSITRAD_GGUF_N_THREADS': '2',
    'ARSITRAD_GGUF_N_BATCH': '256',
    'ARSITRAD_VISION_MODEL_PATH': str(model_path),
    'ARSITRAD_VISION_MMPROJ_PATH': str(mmproj_path),
    'ARSITRAD_VISION_BASE_URL': vision_url,
    'ARSITRAD_VISION_MODEL': os.environ.get('ARSITRAD_VISION_MODEL', 'gemma-4-E4B-it'),
    'ARSITRAD_VISION_MAX_TOKENS': os.environ.get('ARSITRAD_VISION_MAX_TOKENS', '220'),
    'ARSITRAD_VISION_TIMEOUT_SECONDS': os.environ.get('ARSITRAD_VISION_TIMEOUT_SECONDS', '60'),
})

api_url = 'http://127.0.0.1:8000'
api_log = Path('/kaggle/working/arsitrad-api.log')
vision_log = Path('/kaggle/working/arsitrad-vision.log')
backend_tunnel_log = Path('/kaggle/working/arsitrad-api-cloudflared.log')
cloudflared_path = Path('/kaggle/working/cloudflared')
llama_cpp_dir = Path('/kaggle/working/llama.cpp')
url_pattern = re.compile(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com')

def read_url(url: str, timeout: int = 5) -> str:
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return response.read().decode('utf-8', errors='ignore')

def wait_for_url(url: str, label: str, attempts: int = 90, delay: float = 1.0) -> str:
    last_error = None
    for _ in range(attempts):
        try:
            body = read_url(url, timeout=3)
            print(f'{label} ready:', url)
            return body
        except Exception as exc:
            last_error = exc
            time.sleep(delay)
    raise RuntimeError(f'{label} did not become ready at {url}: {last_error}')

def kill_matching_processes(markers):
    ps = subprocess.run(['ps', '-eo', 'pid,args'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT).stdout
    killed = []
    for line in ps.splitlines():
        if not any(marker in line for marker in markers):
            continue
        if 'ipykernel' in line or 'jupyter' in line:
            continue
        parts = line.strip().split(maxsplit=1)
        if not parts or not parts[0].isdigit():
            continue
        pid = int(parts[0])
        if pid == os.getpid():
            continue
        try:
            os.kill(pid, signal.SIGKILL)
            killed.append(pid)
        except ProcessLookupError:
            pass
    if killed:
        print('Killed stale processes:', killed)

def ensure_cloudflared():
    if cloudflared_path.exists():
        return
    print('Downloading cloudflared...')
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared_path)
    cloudflared_path.chmod(cloudflared_path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

def ensure_llama_server() -> str:
    candidates = [
        os.environ.get('ARSITRAD_LLAMA_SERVER_BIN'),
        shutil.which('llama-server'),
        str(llama_cpp_dir / 'build' / 'bin' / 'llama-server'),
        str(llama_cpp_dir / 'build' / 'tools' / 'server' / 'llama-server'),
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return str(Path(candidate))
        if candidate and shutil.which(candidate):
            return str(shutil.which(candidate))

    if os.environ.get('ARSITRAD_BUILD_LLAMA_SERVER', '1') != '1':
        raise RuntimeError('llama-server is missing and ARSITRAD_BUILD_LLAMA_SERVER is disabled.')

    print('llama-server not found; building llama.cpp server for the Gemma vision bridge...')
    if not llama_cpp_dir.exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(llama_cpp_dir)])
    subprocess.check_call(['cmake', '-B', 'build', '-DGGML_CUDA=ON', '-DLLAMA_CURL=OFF'], cwd=llama_cpp_dir)
    subprocess.check_call(['cmake', '--build', 'build', '--config', 'Release', '-j', '2', '--target', 'llama-server'], cwd=llama_cpp_dir)

    built_candidates = [
        llama_cpp_dir / 'build' / 'bin' / 'llama-server',
        llama_cpp_dir / 'build' / 'tools' / 'server' / 'llama-server',
    ]
    for candidate in built_candidates:
        if candidate.exists():
            return str(candidate)
    raise FileNotFoundError('Built llama.cpp, but llama-server binary was not found.')

def start_vision_server():
    try:
        wait_for_url(f'{vision_url}/health', 'Existing Gemma vision bridge', attempts=2, delay=0.5)
        print('Reusing existing Gemma vision bridge:', vision_url)
        return None
    except Exception:
        pass

    if not model_path.exists():
        message = f'Gemma vision model missing: {model_path}. Run the GGUF download cell first.'
        if require_vision:
            raise FileNotFoundError(message)
        print('WARNING:', message)
        return None
    if not mmproj_path.exists():
        message = f'Gemma vision mmproj missing: {mmproj_path}. Run the GGUF/mmproj download cell first.'
        if require_vision:
            raise FileNotFoundError(message)
        print('WARNING:', message)
        return None

    llama_server = ensure_llama_server()
    if vision_log.exists():
        vision_log.unlink()
    vision_cmd = [
        llama_server,
        '-m', str(model_path),
        '--mmproj', str(mmproj_path),
        '--host', '127.0.0.1',
        '--port', '8080',
        '-c', os.environ.get('ARSITRAD_VISION_CTX', '4096'),
        '-ngl', os.environ.get('ARSITRAD_VISION_N_GPU_LAYERS', '999'),
    ]
    print('Starting Gemma vision bridge:', ' '.join(vision_cmd))
    with vision_log.open('w', encoding='utf-8') as log_handle:
        proc = subprocess.Popen(vision_cmd, cwd=repo_root, env=os.environ.copy(), stdout=log_handle, stderr=subprocess.STDOUT)
    wait_for_url(f'{vision_url}/health', 'Gemma vision bridge', attempts=180, delay=2.0)
    return proc

def wait_for_tunnel(log_path: Path, label: str, proc: subprocess.Popen, attempts: int = 60):
    for _ in range(attempts):
        if proc.poll() is not None:
            break
        if log_path.exists():
            log_text = log_path.read_text(encoding='utf-8', errors='ignore')
            match = url_pattern.search(log_text)
            if match:
                return match.group(0)
        time.sleep(2)
    tail = log_path.read_text(encoding='utf-8', errors='ignore')[-4000:] if log_path.exists() else ''
    print(tail)
    raise RuntimeError(f'{label} Cloudflare tunnel failed before a public URL was issued.')

for log_path in [api_log, vision_log, backend_tunnel_log]:
    if log_path.exists():
        log_path.unlink()

kill_matching_processes(['llama-server', '--mmproj', '127.0.0.1:8080'])
subprocess.run("pkill -f 'uvicorn api.server:app' || true", shell=True, check=False)
subprocess.run("pkill -f 'cloudflared tunnel --url http://127.0.0.1:8000' || true", shell=True, check=False)

vision_proc = start_vision_server()

# Launch command equivalent: python -m uvicorn api.server:app --host 0.0.0.0 --port 8000
env = os.environ.copy()
env['PYTHONPATH'] = str(repo_root) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
api_cmd = [
    sys.executable,
    '-m',
    'uvicorn',
    'api.server:app',
    '--host',
    '0.0.0.0',
    '--port',
    '8000',
]

with api_log.open('w', encoding='utf-8') as log_handle:
    api_proc = subprocess.Popen(api_cmd, cwd=repo_root, env=env, stdout=log_handle, stderr=subprocess.STDOUT)

backend_health_text = wait_for_url(f'{api_url}/health', 'FastAPI backend')
backend_health = json.loads(backend_health_text)
if not backend_health.get('vision_enabled'):
    raise RuntimeError(f'FastAPI backend is not connected to the Gemma vision bridge: {backend_health}')
wait_for_url(f'{api_url}/api/bootstrap', 'FastAPI bootstrap')
ensure_cloudflared()

backend_tunnel_proc = subprocess.Popen(
    [
        str(cloudflared_path),
        'tunnel',
        '--url',
        api_url,
        '--no-autoupdate',
        '--logfile',
        str(backend_tunnel_log),
    ],
    cwd='/kaggle/working',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

backend_public_url = wait_for_tunnel(backend_tunnel_log, 'backend API', backend_tunnel_proc)
os.environ['ARSITRAD_BACKEND_TUNNEL_URL'] = backend_public_url

print('Gemma vision bridge local URL:', vision_url)
print('Gemma vision PID:', vision_proc.pid if vision_proc else 'reused existing server')
print('FastAPI backend local URL:', api_url)
print('FastAPI backend public URL:', backend_public_url)
print('FastAPI PID:', api_proc.pid)
print('Backend tunnel PID:', backend_tunnel_proc.pid)
print('Vision log:', vision_log)
print('API log:', api_log)
print('Backend tunnel log:', backend_tunnel_log)

## Launch production Next.js frontend and expose the AI Advisor demo URL

Run this after the Gemma vision bridge + FastAPI backend + backend tunnel cell. It starts the production Next.js workbench with the public backend URL baked into `NEXT_PUBLIC_API_BASE_URL`, then exposes the frontend through a second Cloudflare tunnel.

Important notes:
- Do **not** reset the whole Kaggle session for frontend issues; keep the backend/model/vision bridge warm and relaunch only the frontend.
- The backend root `/` may return `404 Not Found`; that is normal. This notebook verifies `/health` and `/api/bootstrap`.
- `/health` must report `vision_enabled=true`; otherwise the Building Doctor image flow is metadata-only and not judge-demo ready.
- The frontend cell kills stale `next dev` / Turbopack processes, clears `.next`, builds production assets, then runs the production server.
- No legacy Streamlit process is launched in the background.

In [ ]:
import json
import os
import re
import signal
import stat
import subprocess
import time
import urllib.request
from pathlib import Path

repo_root = Path('/kaggle/working/arsitrad')
web_root = repo_root / 'web'
os.chdir(repo_root)
frontend_url = 'http://127.0.0.1:3000'
frontend_log = Path('/kaggle/working/arsitrad-next.log')
frontend_tunnel_log = Path('/kaggle/working/arsitrad-web-cloudflared.log')
cloudflared_path = Path('/kaggle/working/cloudflared')
url_pattern = re.compile(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com')

backend_public_url = globals().get('backend_public_url') or os.environ.get('ARSITRAD_BACKEND_TUNNEL_URL')
if not backend_public_url:
    raise RuntimeError('Run the FastAPI backend + backend tunnel cell first; backend_public_url is missing.')

def read_url(url: str, timeout: int = 5) -> str:
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return response.read().decode('utf-8', errors='ignore')

def wait_for_url(url: str, label: str, attempts: int = 90, delay: float = 1.0) -> str:
    last_error = None
    for _ in range(attempts):
        try:
            body = read_url(url, timeout=3)
            print(f'{label} ready:', url)
            return body
        except Exception as exc:
            last_error = exc
            time.sleep(delay)
    raise RuntimeError(f'{label} did not become ready at {url}: {last_error}')

def ensure_cloudflared():
    if cloudflared_path.exists():
        return
    print('Downloading cloudflared...')
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared_path)
    cloudflared_path.chmod(cloudflared_path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

def wait_for_tunnel(log_path: Path, label: str, proc: subprocess.Popen, attempts: int = 60):
    for _ in range(attempts):
        if proc.poll() is not None:
            break
        if log_path.exists():
            log_text = log_path.read_text(encoding='utf-8', errors='ignore')
            match = url_pattern.search(log_text)
            if match:
                return match.group(0)
        time.sleep(2)
    tail = log_path.read_text(encoding='utf-8', errors='ignore')[-4000:] if log_path.exists() else ''
    print(tail)
    raise RuntimeError(f'{label} Cloudflare tunnel failed before a public URL was issued.')

def kill_matching_processes(markers):
    ps = subprocess.run(['ps', '-eo', 'pid,args'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT).stdout
    killed = []
    for line in ps.splitlines():
        if not any(marker in line for marker in markers):
            continue
        if 'ipykernel' in line or 'jupyter' in line:
            continue
        parts = line.strip().split(maxsplit=1)
        if not parts or not parts[0].isdigit():
            continue
        pid = int(parts[0])
        if pid == os.getpid():
            continue
        try:
            os.kill(pid, signal.SIGKILL)
            killed.append(pid)
        except ProcessLookupError:
            pass
    if killed:
        print('Killed stale frontend processes:', killed)
    else:
        print('No stale frontend processes found.')

# Backend root `/` returning 404 is expected. Verify real API endpoints instead.
# TryCloudflare hostnames can take a minute to propagate inside Kaggle DNS, so retry instead of failing on first gaierror.
print('Checking backend API:', backend_public_url)
backend_health_text = wait_for_url(f'{backend_public_url}/health', 'Backend public /health', attempts=120, delay=2.0)
backend_health = json.loads(backend_health_text)
print('/health:', backend_health_text[:500])
print('AI Advisor / Building Doctor vision status:', 'Gemma vision' if backend_health.get('vision_enabled') else 'metadata-only', backend_health.get('vision_base_url'))
if not backend_health.get('vision_enabled'):
    raise RuntimeError(f'Backend public URL is metadata-only; Building Doctor vision is not enabled: {backend_health}')
print('/api/bootstrap:', wait_for_url(f'{backend_public_url}/api/bootstrap', 'Backend public /api/bootstrap', attempts=120, delay=2.0)[:500])

for log_path in [frontend_log, frontend_tunnel_log]:
    if log_path.exists():
        log_path.unlink()

kill_matching_processes([
    'next dev',
    'next start',
    'next-server',
    'turbopack',
    str(web_root),
])
kill_matching_processes(['cloudflared tunnel --url http://127.0.0.1:3000'])
time.sleep(2)

next_build_dir = web_root / '.next'
if next_build_dir.exists():
    subprocess.run(['rm', '-rf', str(next_build_dir)], check=True)
    print('Removed stale .next build directory.')

if not (web_root / 'node_modules').exists():
    print('Installing Next.js dependencies with npm ci...')
    subprocess.check_call(['npm', 'ci', '--no-audit', '--no-fund'], cwd=web_root)

env = os.environ.copy()
env['NEXT_PUBLIC_API_BASE_URL'] = backend_public_url

print('Building production Next.js frontend...')
# Equivalent shell command: npm run build
subprocess.check_call(['npm', 'run', 'build'], cwd=web_root, env=env)

with frontend_log.open('w', encoding='utf-8') as log_handle:
    frontend_proc = subprocess.Popen(
        ['npx', 'next', 'start', '-H', '0.0.0.0', '-p', '3000'],
        cwd=web_root,
        env=env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )

html = wait_for_url(frontend_url, 'Next.js frontend')
bad_markers = ['hmr-client', 'next-devtools', 'next dev']
found_bad = [marker for marker in bad_markers if marker in html.lower()]
if found_bad:
    tail = frontend_log.read_text(encoding='utf-8', errors='ignore')[-4000:] if frontend_log.exists() else ''
    print(tail)
    raise RuntimeError(f'Still serving dev build markers: {found_bad}. A stale dev server is probably still owning port 3000.')
print('Production frontend verified. No dev/HMR markers found in localhost HTML.')

ensure_cloudflared()
frontend_tunnel_proc = subprocess.Popen(
    [
        str(cloudflared_path),
        'tunnel',
        '--url',
        frontend_url,
        '--no-autoupdate',
        '--logfile',
        str(frontend_tunnel_log),
    ],
    cwd='/kaggle/working',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

frontend_public_url = wait_for_tunnel(frontend_tunnel_log, 'frontend', frontend_tunnel_proc)
os.environ['ARSITRAD_FRONTEND_TUNNEL_URL'] = frontend_public_url

print('Arsitrad Next.js public demo URL:', frontend_public_url)
print('AI Advisor / Building Doctor demo is live with Gemma visual triage enabled.')
print('Backend API public URL:', backend_public_url)
print('Next.js local URL:', frontend_url)
print('Next.js PID:', frontend_proc.pid)
print('Frontend tunnel PID:', frontend_tunnel_proc.pid)
print('Next.js log:', frontend_log)
print('Frontend tunnel log:', frontend_tunnel_log)
print('If buttons are dead, inspect the new URL for dev markers before resetting Kaggle. Resetting the full session should be the last resort.')


## When to use which notebook

Use `Arsitrad_v2_Kaggle_Demo.ipynb` when you want:
- lowest setup friction
- cleaner live demo
- safer judge flow when you do not need the full technical proof

Use `Arsitrad-Full.ipynb` when you want:
- the full sequential technical run
- ingestion / retrieval inspection
- mandatory GGUF generation
- Gemma `mmproj` download and visual triage bridge
- FastAPI backend verification
- Next.js AI Advisor / Building Doctor workbench verification
- Cloudflare public URL exposure from inside Kaggle

Legacy Streamlit is intentionally not launched by this notebook. If you need rollback outside the notebook, use `./scripts/run_legacy_streamlit.sh`.